In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

from rich.pretty import pprint

import numpy as np
import cv2

import panel as pn

from enderscope.image import to_pil, crop_image, load_image
from enderscope.qr_codes import get_qr_viz

In [ ]:
pn.extension("plotly", "ipywidgets")

In [ ]:
PATH_2_IMAGES = Path(".").joinpath("qr_resources", "images", "qr_codes")
PATH_2_IMAGES

In [ ]:
def get_qr_path(name: str, index=int):
    return (
        str(PATH_2_IMAGES.joinpath(f"qr_{name}_{index}.jpg"))
        if isinstance(name, str)
        else str(name)
    )


def load_qr_image(
    name: str, index: int, rgb: bool = True, image_size: int = None
) -> np.ndarray:
    return load_image(
        image_path=get_qr_path(name, index), rgb=rgb, image_size=image_size
    )

In [ ]:
image = crop_image(load_qr_image(name="large", index=1, image_size=2592))
to_pil(image)

In [ ]:
image = load_qr_image(name="large", index=1, image_size=2592)

qcd = cv2.QRCodeDetector()

retval, decoded_info, points, _ = qcd.detectAndDecodeMulti(image)
retval, decoded_info

pprint({"succes":retval, "value": decoded_info})

if retval:
    for point in points[0]:
        cv2.circle(image,[int(c) for c in point], 20, (0,0,255), -1)


to_pil(image=image).resize((600, 600))

In [ ]:
for d in [
    2,
    4,
    # 6,
    # 8,
    # 10,
]:
    image = load_qr_image(name="large", index=1, image_size=2592 // d)

    qcd = cv2.QRCodeDetector()

    retval, decoded_info, points, _ = qcd.detectAndDecodeMulti(image)
    retval, decoded_info

    pprint({"succes": retval, "value": decoded_info, "factor": d, "size": 2592 // d})

    if retval:
        for point in points[0]:
            cv2.circle(image, [int(c) for c in point], 6, (0, 0, 255), -1)
    else:
        break

to_pil(image=image)  # .resize((600, 600))

In [ ]:
ret_data = []

for i in [1, 2, 3, 4, 5, 6]:
    image = load_qr_image(name="small", index=i)
    crop_top = 100
    crop_bottom = 300
    crop_left = 50
    crop_right = 300

    # image = image[
    #     2592 // 2 - crop_top : 2592 // 2 + crop_bottom,
    #     2592 // 2 - crop_left : 2592 // 2 + crop_right,
    # ]

    # image = cv2.detailEnhance(image, sigma_s=10, sigma_r=0.15)
    # image = cv2.GaussianBlur(image, (5, 5), 0)
    # image = cv2.medianBlur(image, 5)
    image = cv2.filter2D(image, -1, np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]]))
    # image = cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)

    retval, decoded_info, points, _ = cv2.QRCodeDetector().detectAndDecodeMulti(image)

    if retval:
        for point in points[0]:
            cv2.circle(image, [int(c) for c in point], 20, (0, 0, 255), -1)

    ret_data.append(
        {"index": i, "succes": retval, "value": decoded_info, "image": image}
    )

pn.GridBox(
    *[
        pn.Column(
            pn.Row(d["index"], d["succes"], d["value"]),
            to_pil(d["image"]).resize((600, 600)),
        )
        for d in ret_data
    ],
    ncols=3,
)

In [ ]:
ret_data = []

for i in list(range(10)):
    image = load_qr_image(name="large", index=i+1,image_size=None)

    # crop_top = 400
    # crop_bottom = 400
    # crop_left = 400
    # crop_right = 400

    # image = image[
    #     2592 // 2 - crop_top : 2592 // 2 + crop_bottom,
    #     2592 // 2 - crop_left : 2592 // 2 + crop_right,
    # ]
    # image = cv2.filter2D(image, -1, np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]]))

    retval, decoded_info, points, _ = cv2.QRCodeDetector().detectAndDecodeMulti(image)

    if retval:
        for point in points[0]:
            cv2.circle(image, [int(c) for c in point], 30, (0, 0, 255), -1)

    ret_data.append(
        {"index": i, "succes": retval, "value": decoded_info, "image": image}
    )

pn.GridBox(
    *[
        pn.Column(
            pn.Row(d["index"], d["succes"], d["value"]),
            to_pil(d["image"]).resize((400, 400)),
        )
        for d in ret_data
    ],
    ncols=5,
)

In [ ]:
to_pil(load_qr_image(name="large", index=5 + 1))

In [ ]:
pn.GridBox(
    *[
        to_pil(get_qr_viz(get_qr_path(name="large", index=i + 1), size=None)).resize(
            (400, 400)
        )
        for i in range(10)
    ],
    ncols=5,
)

In [ ]:
pn.GridBox(
    *[
        to_pil(get_qr_viz(get_qr_path(name="large", index=i + 1), size=None)).resize(
            (500, 500)
        )
        for i in range(8)
    ],
    ncols=4,
)

In [ ]:
image = load_qr_image(name="multi", index=7, image_size=None)
image = crop_image(image, crop_top=800, crop_bottom=600, crop_left=500, crop_right=800)
image = cv2.rotate(image, cv2.ROTATE_180)
# to_pil(image=image)
to_pil(get_qr_viz(image_object=image, size=None))
# data
# data["points"][0]